In [1]:
import os
import sys
import pandas as pd
import numpy as np
from scipy import stats


spend = pd.DataFrame({
    'customer_id': [f'C{str(i).zfill(3)}' for i in range(1,21)],
    'product': ['Standard']*10 + ['Premium']*10,
    'monthly_spend': [420,390,510,470,380,430,460,490,410,440,
                      980,1150,870,1320,1050,1200,940,1080,1410,990],
    'credit_score':  [680,710,690,720,660,700,715,695,705,685,
                      760,790,750,810,775,800,765,780,820,755],
    'age': [28,34,45,30,52,41,37,29,48,33,44,38,55,31,47,36,42,50,29,39]
})

loans = pd.DataFrame({
    'app_id': [f'L{str(i).zfill(3)}' for i in range(1,11)],
    'region': ['North']*5 + ['South']*5,
    'loan_amount': [15000,22000,8000,18000,12000,30000,25000,9000,27000,11000],
    'approved': [1,1,0,1,0,1,1,0,1,0],
    'processing_days': [3,5,7,4,6,4,3,8,5,9]
})

In [2]:
print(spend["monthly_spend"].mean())
print(spend["monthly_spend"].median())
print(spend["monthly_spend"].std())

769.5
690.0
359.27522070207726


In [3]:
print(spend.describe())
range_1 = spend["credit_score"].max() - spend["credit_score"].min()
print("=========")
print(range_1)
print("=========")
sorted_1 = spend.sort_values("credit_score",ascending = False)
print(sorted_1[0:1])

       monthly_spend  credit_score        age
count      20.000000     20.000000  20.000000
mean      769.500000    738.250000  39.400000
std       359.275221     48.020692   8.280605
min       380.000000    660.000000  28.000000
25%       437.500000    698.750000  32.500000
50%       690.000000    735.000000  38.500000
75%      1057.500000    776.250000  45.500000
max      1410.000000    820.000000  55.000000
160
   customer_id  product  monthly_spend  credit_score  age
18        C019  Premium           1410           820   29


In [4]:
premium = spend[spend["product"] == "Premium"]
standard = spend[spend["product"] == "Standard"]

print(premium.head(2))
print(standard.head(2))

   customer_id  product  monthly_spend  credit_score  age
10        C011  Premium            980           760   44
11        C012  Premium           1150           790   38
  customer_id   product  monthly_spend  credit_score  age
0        C001  Standard            420           680   28
1        C002  Standard            390           710   34


In [5]:
variation = spend.groupby("product")["monthly_spend"].agg(average = 'mean', deviation ='std')
variation["cv"] = variation["deviation"]/variation["average"]
print(variation)

          average   deviation        cv
product                                
Premium    1099.0  171.558218  0.156104
Standard    440.0   42.426407  0.096424


In [6]:
premium = spend[spend["product"] == "Premium"]["monthly_spend"]
standard = spend[spend["product"] == "Standard"]["monthly_spend"]

t_stat, pval = stats.ttest_ind(standard, premium)
print(f"t= {t_stat:.2f}, p = {pval:.2f}")

if pval <0.05:
    print("Reject Null Hypothesis: There is a significant statistical difference in null standard and premium")

t= -11.79, p = 0.00
Reject Null Hypothesis: There is a significant statistical difference in null standard and premium


In [7]:
north = loans[loans["region"]== "North"]["processing_days"]
south = loans[loans["region"]== "South"]["processing_days"]

avg_north = north.mean()
avg_south = south.mean()

print(avg_north)
print(avg_south)

tstat, p_val = stats.ttest_ind(north, south)

print(f"t = {tstat:.2f}, p = {p_val:.2f}")

if p_val<0.05:
    print("Reject Null Hypothesis: There is a significant difference in processing days")
else:
    print("Fail to reject Null Hypothesis: There is no significant difference in processing days")

5.0
5.8
t = -0.59, p = 0.57
Fail to reject Null Hypothesis: There is no significant difference in processing days


In [8]:
spend.to_csv(r"C:\Users\chest\OneDrive\Desktop\Capital One\spend.csv",index = False, encoding = "utf-8")

In [9]:
loans.to_csv(r"C:\Users\chest\OneDrive\Desktop\Capital One\loans.csv",index = False, encoding = "utf-8")

In [10]:
spend.describe()

,monthly_spend,credit_score,age
count,20.000000,20.000000,20.000000
mean,769.500000,738.250000,39.400000
std,359.275221,48.020692,8.280605
min,380.000000,660.000000,28.000000
25%,437.500000,698.750000,32.500000
50%,690.000000,735.000000,38.500000
75%,1057.500000,776.250000,45.500000
max,1410.000000,820.000000,55.000000


In [12]:
Q1 = spend["monthly_spend"].quantile(0.25)
Q3 = spend["monthly_spend"].quantile(0.75)

IQR =  Q3-Q1

print(Q1)
print(Q3)
print(IQR)

IQR_low = Q1  - 1.5*IQR
IQR_high = Q3 + 1.5*IQR

print(IQR_low,IQR_high)

437.5
1057.5
620.0
-492.5 1987.5


In [ ]:
standard_scores = spend[spend["product"]=="Standard"]["credit_score"]
print(f"standard score for population: {standard_scores.mean()}")

standar score for population: 696.0


In [24]:
ttest, ptwo = stats.ttest_1samp(standard_scores, popmean= 700)
pone = ptwo/2

print(ttest,pone)

-0.6998542122237651 0.25085292030908324


In [30]:
loan_north = loans[loans["region"]=="North"]["loan_amount"]
loan_south  =loans[loans["region"]=="South"]["loan_amount"]

print(loan_north.head())

ttest_1, pval1 = stats.ttest_ind(loan_north,loan_south)

print(ttest_1, pval1)

if pval1 < 0.05:
    print("Reject null hypothesis: Loan amounts are not the same/similar")
else:
    print("Fail to reject null: Loan amounts are same")

0    15000
1    22000
2     8000
3    18000
4    12000
Name: loan_amount, dtype: int64
-1.0896313215662032 0.30760613879149823
Fail to reject null: Loan amounts are same


In [32]:
for region in ['North','South']:
    group = loans[loans['region']==region]
    rate = group['approved'].mean()
    print(f"{region}: approval rate {rate:.0%}, avg loan ${group['loan_amount'].mean():,.0f}")

North: approval rate 60%, avg loan $15,000
South: approval rate 60%, avg loan $20,400
